# Telecom Credit Scoring Nigeria — classification de `default_risk`

Ce notebook charge la base CSV, inspecte et nettoie les données, entraîne trois modèles de classification, les évalue sur un jeu de test indépendant, compare leurs scores et sauvegarde automatiquement le meilleur modèle.

**Cible à prédire :** `default_risk` (`low`, `medium`, `high`).

**Précaution contre la fuite de données :** les colonnes `credit_rating`, `loan_eligibility` et `max_loan_amount_ngn` sont exclues des variables explicatives car elles peuvent être calculées à partir du score ou du risque. Les identifiants, numéros de téléphone et la date unique d'évaluation sont également exclus. Le choix final des variables doit être validé avec les règles métier réelles.

**Fichier attendu :** `telecom_credit_scoring_nigeria.csv`, dans le même dossier que ce notebook. Les fichiers de sortie seront créés dans le dossier courant.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Bibliothèques chargées.')

## 1. Charger la dataset

In [ ]:
DATA_PATH = 'telecom_credit_scoring_nigeria.csv'
if not os.path.exists(DATA_PATH) and os.path.exists('/mnt/data/telecom_credit_scoring_nigeria.csv'):
    DATA_PATH = '/mnt/data/telecom_credit_scoring_nigeria.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset chargée : {DATA_PATH}')
print(f'Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
display(df.head())

## 2. Explorer et diagnostiquer les données

In [ ]:
display(df.info())
display(df.describe(include='all').T)
print('\nValeurs manquantes par colonne :')
display(df.isna().sum().sort_values(ascending=False).to_frame('valeurs_manquantes'))
print('\nDoublons de lignes :', df.duplicated().sum())

if 'default_risk' not in df.columns:
    raise ValueError("La colonne cible 'default_risk' est absente de la dataset.")

print('\nRépartition de la cible :')
display(df['default_risk'].value_counts(dropna=False).to_frame('effectif'))
display((df['default_risk'].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame('pourcentage'))

## 3. Nettoyage et préparation

On supprime les lignes sans cible, les doublons exacts, puis on sépare la cible des variables explicatives. Les valeurs manquantes des variables seront imputées dans le pipeline, afin que le même traitement soit appliqué pendant l'entraînement et lors de futures prédictions.

In [ ]:
df = df.drop_duplicates().copy()
df = df.dropna(subset=['default_risk']).copy()
df['default_risk'] = df['default_risk'].astype(str).str.strip().str.lower()
df = df[df['default_risk'] != ''].copy()

# Colonnes exclues : identifiants, date constante et variables potentiellement dérivées de la cible.
columns_to_drop = [
    'default_risk', 'customer_id', 'phone_number', 'assessment_date',
    'credit_rating', 'loan_eligibility', 'max_loan_amount_ngn'
]
X = df.drop(columns=[c for c in columns_to_drop if c in df.columns])
y = df['default_risk']

# Détecter automatiquement les variables numériques et catégorielles.
numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print('Dimensions après nettoyage :', df.shape)
print('Variables numériques :', numeric_features)
print('Variables catégorielles :', categorical_features)
print('\nClasses à prédire :', sorted(y.unique()))
display(X.head())

## 4. Séparer les données en entraînement et test

Le jeu de test représente 20 % des données. `stratify=y` conserve approximativement la répartition des classes dans les deux ensembles. Le jeu de test ne sert pas à entraîner les modèles.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Entraînement : {X_train.shape[0]:,} lignes')
print(f'Test         : {X_test.shape[0]:,} lignes')
print('\nRépartition des classes dans y_train :')
display((y_train.value_counts(normalize=True) * 100).round(2).to_frame('pourcentage'))

## 5. Définir le prétraitement et les trois modèles

- **Régression logistique** : modèle de référence, généralement rapide.
- **Random Forest** : ensemble d'arbres de décision.
- **Extra Trees** : ensemble d'arbres avec une randomisation supplémentaire.

Le score principal pour choisir le meilleur modèle sera le **F1 macro**, qui donne la même importance à chacune des classes. L'accuracy seule peut masquer de mauvaises performances sur une classe moins fréquente.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

transformers = []
if numeric_features:
    transformers.append(('num', numeric_transformer, numeric_features))
if categorical_features:
    transformers.append(('cat', categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

models = {
    'Régression logistique': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=24, min_samples_leaf=2,
        class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=150, max_depth=24, min_samples_leaf=2,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
    )
}
print('Trois modèles définis.')

## 6. Entraîner, tester et comparer les modèles

In [ ]:
trained_models = {}
results = []

for name, estimator in models.items():
    print(f'\n===== Entraînement : {name} =====')
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', estimator)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = {
        'Modèle': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Précision macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Rappel macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1 macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'F1 pondéré': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    }
    results.append(metrics)
    trained_models[name] = pipeline
    print(pd.Series(metrics).to_string())

    print('\nRapport de classification :')
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results).sort_values('F1 macro', ascending=False).reset_index(drop=True)
display(results_df.style.format({c: '{:.4f}' for c in results_df.columns if c != 'Modèle'}))
results_df.to_csv('comparaison_modeles.csv', index=False)
print('Comparaison enregistrée dans comparaison_modeles.csv')

## 7. Visualiser la comparaison

In [ ]:
ax = results_df.set_index('Modèle')[['Accuracy', 'F1 macro', 'F1 pondéré']].plot(
    kind='bar', figsize=(11, 6), ylim=(0, 1)
)
ax.set_title('Comparaison des modèles sur le jeu de test')
ax.set_ylabel('Score')
ax.set_xlabel('Modèle')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Évaluer en détail le meilleur modèle

In [ ]:
best_name = results_df.iloc[0]['Modèle']
best_model = trained_models[best_name]
best_predictions = best_model.predict(X_test)
class_names = sorted(y.unique())

print('Meilleur modèle selon le F1 macro :', best_name)
print('\nRapport de classification final :')
print(classification_report(y_test, best_predictions, labels=class_names, zero_division=0))

cm = confusion_matrix(y_test, best_predictions, labels=class_names)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(values_format='d', cmap='Blues')
plt.title(f'Matrice de confusion — {best_name}')
plt.tight_layout()
plt.savefig('matrice_confusion_meilleur_modele.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Sauvegarder le meilleur modèle

Le pipeline complet est sauvegardé, prétraitement inclus. Il pourra ainsi être réutilisé sans refaire manuellement l'encodage ou la normalisation.

In [ ]:
MODEL_PATH = 'meilleur_modele_credit_scoring.joblib'
joblib.dump(best_model, MODEL_PATH)

metadata = {
    'nom_modele': best_name,
    'cible': 'default_risk',
    'variables': X.columns.tolist(),
    'classes': class_names,
    'metrique_selection': 'F1 macro',
    'score_f1_macro_test': float(results_df.iloc[0]['F1 macro'])
}
joblib.dump(metadata, 'metadata_meilleur_modele.joblib')

print(f'Modèle sauvegardé : {MODEL_PATH}')
print('Métadonnées sauvegardées : metadata_meilleur_modele.joblib')
print('\nFichiers générés :')
for filename in [MODEL_PATH, 'metadata_meilleur_modele.joblib', 'comparaison_modeles.csv',
                 'comparaison_modeles.png', 'matrice_confusion_meilleur_modele.png']:
    print(f'- {filename}: {os.path.getsize(filename):,} octets' if os.path.exists(filename) else f'- {filename}: absent')

## 10. Tester une prédiction avec le modèle sauvegardé

Exemple : on recharge le modèle enregistré et on prédit le risque des premières lignes du jeu de test. Les colonnes fournies doivent être les mêmes que celles utilisées lors de l'entraînement.

In [ ]:
loaded_model = joblib.load('meilleur_modele_credit_scoring.joblib')
exemples = X_test.head(5)
predictions = loaded_model.predict(exemples)
resultats_predictions = exemples.copy()
resultats_predictions['risque_predit'] = predictions
resultats_predictions['risque_reel'] = y_test.loc[exemples.index].values
display(resultats_predictions)
print('Chargement et prédiction du modèle sauvegardé : OK')

## 11. Conclusion

Le tableau de comparaison indique quel modèle obtient le meilleur F1 macro sur le jeu de test. Reporte ici ses scores réels après exécution du notebook : ne saisis pas de scores avant d'avoir exécuté le code.
